In [1]:
def fetch_json_data(file_path, output_file_path="cust_data.json"):
    """
    Fetches recipe data from a local JSON file, processes it, and returns only the relevant fields.
    """
    try:
        with open(file_path, 'r') as file:
            data = json.load(file)

        all_recipes = []

        # Iterate over each entry in the JSON file
        for entry in data:
            # Skip if entry is not a dictionary
            if not isinstance(entry, dict):
                print(f"Skipping non-dictionary entry: {entry}")
                continue

            recipe_info_list = entry.get("recipe_info", [])  # Ensure it's a list

            # Ensure recipe_info_list is actually a list before proceeding
            if not isinstance(recipe_info_list, list):
                print(f"Skipping entry due to unexpected format: {entry}")
                continue

            # Extract delivery date - now much simpler
            delivery_date = None
            entry_id = entry.get("_id", {})
            if isinstance(entry_id, dict):
                delivery_date = entry_id.get("delivery_date")
                # You might want to validate the date format here if needed

            # Iterate through each recipe_info dictionary in the list
            for recipe_info in recipe_info_list:
                # Skip if recipe_info is not a dictionary
                if not isinstance(recipe_info, dict):
                    print(f"Skipping non-dictionary recipe_info: {recipe_info}")
                    continue

                # Extract ingredients and merge with variant ingredients
                ingredients = recipe_info.get("ingredients", [])
                variant_ingredients = []

                # Process the variants if available
                variants = recipe_info.get("variants", {})
                if isinstance(variants, dict):  # Ensure "variants" is a dictionary
                    variant_ingredients = variants.get("variant_ingredients", [])

                # Merge ingredients from both the recipe_info and variants
                all_ingredients = list(set(ingredients + variant_ingredients))

                # Prepare the recipe data with only the necessary fields
                recipe = {
                    "dish_name": recipe_info.get("dish_name"),
                    "meal_category": recipe_info.get("meal_category"),
                    "description": recipe_info.get("description"),
                    "cuisine": recipe_info.get("cuisine"),
                    "ingredients": all_ingredients,  # Merged ingredients
                    "allergens_contain": recipe_info.get("allergens_contain", []),
                    "meal_type": entry.get("meal_type"),
                    "spice_level": recipe_info.get("spice_level", ""),
                    "is_auto_select": recipe_info.get("is_auto_select"),
                    "rating": recipe_info.get("rating") if "rating" in recipe_info else None,
                    "delivery_date": delivery_date  # Use the date string directly
                }

                # Add the processed recipe data to the list
                all_recipes.append(recipe)

        # If an output file path is provided, save the processed data to that file
        if output_file_path and all_recipes:
            with open(output_file_path, 'w') as output_file:
                json.dump(all_recipes, output_file, indent=4)
            print(f"Processed data saved to {output_file_path}")

        return all_recipes

    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return []
    except json.JSONDecodeError:
        print(f"Error: Failed to decode JSON data in file '{file_path}'. Please check the JSON syntax.")
        return []
    except Exception as e:
        print(f"Error processing file {file_path}: {str(e)}")
        return []

In [4]:
import json
input_file_path = "64a3930ab0d7b763fc053648.json"
output_file_path = "processed_64a3930ab0d7b763fc053648.json"

processed_data = fetch_json_data(input_file_path, output_file_path)


Processed data saved to processed_64a3930ab0d7b763fc053648.json


In [5]:
import pandas as pd

import json
from IPython.display import display, Markdown

def process_and_analyze_json(output_file_path="processed_data.json"):
    """
    Process and analyze the recipe data from a JSON file.
    It will:
    - Extract and merge the ingredients.
    - Analyze top cuisines, spice levels, and user selections.
    - Display visualizations like pie charts.
    - List all user-rated recipes sorted from highest to lowest rating.
    """
    try:
        # Load the processed JSON data directly
        with open(output_file_path, "r") as file:
            recipes = json.load(file)

        # Convert the processed data into a DataFrame
        df = pd.DataFrame(recipes)

        # Check if 'cuisine' column exists
        if 'cuisine' not in df.columns:
            raise KeyError("Missing 'cuisine' column in the dataset")

        # Identify the top cuisine preference
        top_cuisine = df['cuisine'].mode()
        # print(f"Top Cuisine: {top_cuisine}")
        
        # Identify the user's most preferred spice level
        top_spice_level = df['spice_level'].mode()[0]
        # print(f"Top Spice Level: {top_spice_level}")

        # No longer filtering by `is_auto_select`
        user_selected_meals = df.copy()  

        # Display a summary of all meals
        user_selected_meals_summary = user_selected_meals.describe(include='object')
        # print("Summary of All Selected Meals:")
        # display(user_selected_meals_summary)

        # Get the most frequent dish names
        top_dish_names = user_selected_meals['dish_name'].value_counts().reset_index()
        top_n = 5
        # display(Markdown("### Most Frequently Selected Dishes:"))
        # display(top_dish_names.head(top_n))

        # Get the count of each cuisine selected by the user
        cuisine_counts = user_selected_meals['cuisine'].value_counts().reset_index()
        cuisine_counts.columns = ["Cuisine", "Count"]
        # display(Markdown("### Total Cuisine Counts Across all Weeks :"))
        # display(cuisine_counts)

        # Display all user-rated recipes sorted from highest to lowest rating
        if 'rating' in df.columns:
            top_rated_recipes = df[df['rating'].notna()].sort_values(by='rating', ascending=False)
            # display(Markdown("### User-Rated Recipes (Sorted by Rating):"))
            # display(top_rated_recipes[['dish_name', 'rating', 'cuisine', 'spice_level', 'delivery_date']])

        if 'rating' in df.columns:
            least_rated_recipes = df[df['rating'].notna()].sort_values(by='rating', ascending=True)
            # display(Markdown("### User-Rated Recipes (Sorted by Rating):"))
            # display(least_rated_recipes[['dish_name', 'rating', 'cuisine', 'spice_level', 'delivery_date']])

        # Pie Chart for Cuisine Preferences
        # sns.set(style="darkgrid")
        # plt.style.use('dark_background')

        # plt.figure(figsize=(4, 4))  
        # plt.pie(
        #     cuisine_counts['Count'], 
        #     labels=cuisine_counts['Cuisine'], 
        #     autopct='%1.1f%%',  
        #     colors=sns.color_palette("tab20", len(cuisine_counts)), 
        #     startangle=90, 
        #     wedgeprops={'edgecolor': 'none'},  
        #     labeldistance=1.1,  
        #     pctdistance=0.85  
        # )
        # plt.title('Cuisines Selected by User', fontsize=8, color='white')  
        # plt.ylabel('')
        # plt.gcf().patch.set_alpha(0)
        # # display(Markdown("### Cuisine Preference Across all Weeks :"))
        # plt.show()

        # Weekly Cuisine Preferences Analysis
        if 'delivery_date' in df.columns:
            df['delivery_date'] = df['delivery_date'].apply(lambda x: str(x) if isinstance(x, dict) else x)
            df['delivery_date'] = pd.to_datetime(df['delivery_date'], errors='coerce')
        
        min_date = df['delivery_date'].min()
        df['week_number'] = df['delivery_date'].apply(lambda x: (x - min_date).days // 7 + 1)

        cuisine_weekly_counts = df.groupby(['week_number', 'cuisine']).size().reset_index(name='count')
        pivot_table = cuisine_weekly_counts.pivot(index="week_number", columns="cuisine", values="count").fillna(0)
        cuisine_order = cuisine_weekly_counts.groupby("cuisine")["count"].sum().sort_values()
        pivot_table = pivot_table[cuisine_order.index]

        # display(Markdown("### Cuisine Preferences over Weeks :"))
        # display(pivot_table)

        # Generate structured query based on analysis
        top_cuisines = ', '.join(df['cuisine'].mode())  # Get top cuisine(s)
        top_spice_level = df['spice_level'].mode()[0]  # Get top spice level
        # Ensure the first column contains dish names
        top_dishes = ', '.join(top_dish_names.iloc[:6, 0])  # ✅ Corrected way

        # Find highest-rated dishes (handling multiple)
        if 'rating' in df.columns and not df['rating'].isna().all():
            max_rating = df['rating'].max()  # Get the highest rating
            top_rated_dishes = df[df['rating'] == max_rating]['dish_name'].unique()  # Get unique top-rated dishes
            top_rated_dishes = ', '.join(top_rated_dishes[:5])  # Limit to top 5 for readability
        else:
            top_rated_dishes = None
        # Find dishes rated less than 3 (User Dislikes)
        user_dislikes = None
        if 'rating' in df.columns and not df['rating'].isna().all():
            disliked_dishes = df[df['rating'] < 3]['dish_name'].unique()  # Get unique low-rated dishes
            user_dislikes = ', '.join(disliked_dishes[:5])  # Limit to 5 for readability

        # Build the query string
        query = f"Spice Level: {top_spice_level}, Cuisine: {top_cuisines}. Popular Dishes: {top_dishes}"

        if top_rated_dishes:
            query += f". Highest Rated Dishes: {top_rated_dishes}"

        # print("Generated Query:")
        # print(query)

        return df, query, user_dislikes  # Return DataFrame and Query


    except FileNotFoundError:
        print(f"Error: The file '{output_file_path}' was not found.")
        return []
    except KeyError as e:
        print(f"Error: {str(e)}")
        return []
    except json.JSONDecodeError:
        print(f"Error: Failed to decode JSON data in file '{output_file_path}'. Please check the JSON syntax.")
        return []
    except Exception as e:
        print(f"Error: {str(e)}")
        return []

In [6]:
df, query, user_dislikes = process_and_analyze_json("processed_64a3930ab0d7b763fc053648.json")  # Call function for each file


In [ ]:
import os
from dotenv import load_dotenv
from langchain.vectorstores import Pinecone as LangChainPinecone
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai
import json
# from recipe_filter import filter_allergens_in_variants, filter_and_sort_recipes
import ast

# Load environment variables
load_dotenv(override=True)

gemini_api_key = os.getenv('GOOGLE_API_KEY')

# Ensure your Google API key is set
genai.configure(api_key=gemini_api_key)

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name = "recipes-fin"

# Load the embedding model (same as used for storing data)
embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Connect Pinecone to LangChain
vectorstore = LangChainPinecone(pc.Index(index_name), embed_model, text_key="text")

# Initialize ChatGoogleGenerativeAI for gemini-1.5-flash
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# Function to generate responses
def generate_response(prompt):
    model = genai.GenerativeModel("gemini-1.5-flash")
    # Generate content based on the prompt
    response = model.generate_content(prompt)
    return response.text

# Function to filter and sort recipes
def filter_recipes(vectorstore, user_avoid_ingredients, user_dislikes, query, meal_category, size, protein_option, protein_category, top_k):
    pinecone_filter = {
        "meal_category": {"$eq": meal_category},  # Filter for specific meal type
        "size": {"$eq": size},  # Filter for specific size
        # "protein_option": {"$eq": protein_option},  # Filter for specific protein option
        "protein_category": {"$eq": protein_category},  # Filter for specific protein category
        "allergens": {"$nin": list(user_avoid_ingredients)},  # Exclude recipes containing allergens
        "ingredients": {"$nin": list(user_avoid_ingredients)}  # Exclude recipes containing allergens
    }
    # # Add `protein_category` filter only if it's not empty
    # if protein_category:
    #     pinecone_filter["protein_category"] = {"$eq": protein_category}


    # Fetch documents using similarity_search
    docs = vectorstore.similarity_search(
        query=query,
        k=top_k * 3 ,  # Get extra to account for duplicates
        filter=pinecone_filter
    )

    # Merge recipes by recipe_id and combine protein_options
    merged_recipes = {}

    
    for doc in docs:
        metadata = doc.metadata
        recipe_id = metadata.get("recipe_id")
        
        # Skip if no recipe_id exists
        if not recipe_id:
            continue
        
        # Initialize new entry if recipe_id not seen
        if recipe_id not in merged_recipes:
            merged_recipes[recipe_id] = {
                **metadata,  # Copy all metadata
                "protein_option": {metadata.get("protein_option")}  # Start as set
            }
        else:
            # Merge protein_options
            existing = merged_recipes[recipe_id]
            new_protein = metadata.get("protein_option")
            if new_protein:
                existing["protein_option"].add(new_protein)
    
    # Prepare final output (convert sets to lists)
    final_recipes = [
        {
            **data,
            "protein_option": list(data["protein_option"]) if data["protein_option"] else []
        }
        for data in merged_recipes.values()
    ]
    print("final",final_recipes[:top_k])
    return final_recipes[:top_k]

# Function to format the filtered recipes into a structured meal plan prompt
def format_meal_plan_prompt(merged_recipes, query, user_avoid_ingredients, user_likes, user_dislikes, user_pref, meal_types):

        # Normalize meal types by stripping whitespace and converting to lowercase
    normalized_meal_types = {meal.strip().lower() for meal in meal_types}
    
    # Define the standard meal type order (customize as needed)
    STANDARD_ORDER = ['morning_snack', 'breakfast', 'lunch', 'dinner', 'evening_snack']
    
    # Filter and order the meal types based on standard order
    ordered_meal_types = [meal for meal in STANDARD_ORDER 
                         if meal in normalized_meal_types]
    
    prompt = f"""Generate a weekly meal plan in JSON format using ONLY the provided recipes. Follow these rules exactly:

1. Recipe Usage:
- Use recipes exactly as provided - do not modify or create new ones
- Format each meal as: "<Dish Name> - <Selected Protein> - <Cuisine> - <Dish Type>"
- Use ONLY the dish_type field for the last component (never meal_category)
- For recipes with multiple protein options:
    * Ensure protein variety across the week (don't serve chicken 3 days in a row)

3. Meal Diversity:
- Alternate between:
  * Light vs heavy meals (e.g. salad → hearty stew)
  * Different cuisines (don't repeat back-to-back)
  * Cooking methods (grilled, baked, fried, etc.)
- Ensure no two consecutive meals have:
  * The same primary ingredient
  * Similar textures/flavor profiles

2. Meal Assignment:
- Never repeat recipes before all are used once
- Fill all selected meal slots - no empty values
- **Strictly follow meal categories:**
    * Breakfast: only 'breakfast' recipes having meal_category as breakfast
    * Lunch/Dinner: only 'meal' recipes having meal_category as meal
    * evening_snavk/morning_snack: only 'snack' recipes i.e. recipes having meal_category as snack
- Include only these meal types: {meal_types}

3. Daily Structure:
- You MUST include these meal types in EXACTLY this order: {ordered_meal_types}
- Never skip or rearrange these meal types
- Never include meal types not in this list
- Maintain consistent meal types across all days

Output Format: Present the meal plan as a JSON object where each day contains meal types as keys and the formatted meal string as values, like this example for Monday: {{\"Monday\": {{\"breakfast\": \"Dish Name - Protein - Cuisine - Dish Type\", \"lunch\": \"...\"}}}}User Preferences:
- Allergens: {user_avoid_ingredients}
- Likes: {user_likes}
- Dislikes: {user_dislikes}
- Preferred Dishes: {user_pref}
- Selected Meal Types: {meal_types}

Available Recipes:"""
    

    for i, recipe in enumerate(merged_recipes, 1):
        prompt += f"Meal {i}:\n"
        prompt += f" Dish Name: {recipe.get('dish_name', 'Unknown')}\n"
        prompt += f" Description: {recipe.get('description', 'No description')}\n"
        prompt += f" Protein Options: {', '.join(recipe.get('protein_option', []))}\n"  # Changed to handle list
        prompt += f" Dish Type: {', '.join(recipe.get('dish_type', []))}\n"  # Changed to handle list
        prompt += f" Ingredients: {', '.join(recipe.get('ingredients', []))}\n"
        prompt += f" Spice Level: {recipe.get('spice_level', 'Not specified')}\n"
        prompt += f" Cuisine: {recipe.get('cuisine', 'Unknown')}\n"
        prompt += f" Meal Category: {recipe.get('meal_category', 'Unknown')}\n"
    return prompt

# Main function to generate the meal plan
def generate_meal_plan(vectorstore, user_avoid_ingredients, user_dislikes, query, user_likes, user_pref, size, protein_option, protein_category, meal_types):
    # Define mapping of meal types to their respective counts
    meal_counts = {
        "breakfast": 8,
        "snack": 8,  # Each snack type (morning/evening) adds 8
        "meal": 0    # Lunch and Dinner are combined into "meal"
    }

    # Initialize counts
    total_snack_count = 0
    total_meal_count = 0
    fetched_recipes = {}

    # Calculate needed recipes
    if "morning_snack" in meal_types or "evening_snack" in meal_types:
        total_snack_count = meal_counts["snack"] * sum(1 for meal in meal_types if "snack" in meal)

    if "lunch" in meal_types:
        total_meal_count += 10
    if "dinner" in meal_types:
        total_meal_count += 10

    # Initial fetch (with allergens)
    for meal_type in meal_types:
        count = meal_counts.get(meal_type, 0)
        if count > 0:
            meal_size = "standard" if meal_type in ["breakfast", "snack"] else size
            fetched_recipes[meal_type] = filter_recipes(
                vectorstore, user_avoid_ingredients, user_dislikes, query, 
                meal_type, meal_size, "", protein_category, count
            )

    if total_meal_count > 0:
        fetched_recipes["meal"] = filter_recipes(
            vectorstore, user_avoid_ingredients, user_dislikes, query,
            "meal", size, protein_option, protein_category, total_meal_count
        )

    if total_snack_count > 0:
        fetched_recipes["snack"] = filter_recipes(
            vectorstore, user_avoid_ingredients, user_dislikes, query,
            "snack", "standard", "", protein_category, total_snack_count
        )

    # Calculate expected minimums
    expected_breakfast = max(0, meal_counts.get("breakfast", 0) - 4) if "breakfast" in meal_types else 0
    expected_snack = max(0, total_snack_count - (3 if total_snack_count == 8 else 6 if total_snack_count == 16 else 0)) if any("snack" in mt for mt in meal_types) else 0
    expected_meal = max(0, total_meal_count - (2 if total_meal_count == 8 else 7 if total_meal_count == 20 else 0)) if ("lunch" in meal_types or "dinner" in meal_types) else 0
    print("fetched",fetched_recipes)


    # Selective retry (only replaces deficient categories)
    if "breakfast" in meal_types and len(fetched_recipes.get("breakfast", [])) < expected_breakfast:
        print(f"Breakfast shortage ({len(fetched_recipes.get('breakfast', []))}/{expected_breakfast}), retrying without filters...")
        fetched_recipes["breakfast"] = filter_recipes(
            vectorstore, set(), user_dislikes, query,
            "breakfast", "standard", "", protein_category, meal_counts["breakfast"]
        )

    if any("snack" in mt for mt in meal_types) and len(fetched_recipes.get("snack", [])) < expected_snack:
        print(f"Snack shortage ({len(fetched_recipes.get('snack', []))}/{expected_snack}), retrying without filters...")
        fetched_recipes["snack"] = filter_recipes(
            vectorstore, set(), user_dislikes, query,
            "snack", "standard", "", protein_category, total_snack_count
        )

    if ("lunch" in meal_types or "dinner" in meal_types) and len(fetched_recipes.get("meal", [])) < expected_meal:
        print(f"Meal shortage ({len(fetched_recipes.get('meal', []))}/{expected_meal}), retrying without filters...")
        fetched_recipes["meal"] = filter_recipes(
            vectorstore, set(), user_dislikes, query,
            "meal", size, protein_option, protein_category, total_meal_count
        )
    print("fetched",fetched_recipes)

    # Debug output
    print("\nFinal recipe counts:")
    for category in ["breakfast", "snack", "meal"]:
        if category in fetched_recipes:
            print(f"{category.capitalize()}: {len(fetched_recipes[category])}")

    # Generate meal plan
    final_docs = [recipe for recipes in fetched_recipes.values() for recipe in recipes]
    final_prompt = format_meal_plan_prompt(final_docs, query, user_avoid_ingredients, user_likes, user_dislikes, user_pref, meal_types)
    meal_plan = generate_response(final_prompt)

    return meal_plan, final_docs

# Example usage
if __name__ == "__main__":
    # User preferences (replace with dynamic input if needed)
    user_avoid_ingredients = {}  # Example allergens
    # user_dislikes = {
    #     "Cod (white fish)", "Cod Fish", "Cuttlefish", "Fish Sauce",
    #     "Gochujang Paste", "Gochujang Sauce", "Local Wild Fish",
    #     "Nile Perch", "Salmon", "Sea Bass", "Squid", "Tuna",
    #     "White Fish", "Worcestershire Sauce"
    # }  # Example disliked ingredients
    
    # query = "Spice Level: Medium, Cuisine: Mediterranean,European,Comfort Food. Popular Dishes: Classic Chicken Salad, Crudites & Sour Cream Dip, Mini Quiches, Omega Egg Protein Pot"

    # Generate the meal plan
    # generate_meal_plan(vectorstore, user_avoid_ingredients, user_dislikes, query, user_likes, user_pref)

In [8]:
query

'Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Key Lime Yoghurt , Omega Egg Protein Pot , Crudites & Sour Cream Dip, Creamy Quinoa Bowl , Mfarakeh, Breakfast Power Bowl '

In [12]:
import time

user_pref = "Mediterranean"
user_likes = "Herb Mashed Potato, Baked Protein & Mashed Potato, Musakhan & Cauli Rice, Thai Mango Salad, Balkan Mushroom Rice, Leek & Potato Mash. Highest Rated Dishes: Balkan Mushroom Rice, Leek & Potato Mash, Chicken a la King & Roasted Sweet Potato, Burrito Bowl"
size = "extra_large"
protein_category = "low"
protein_option = ""
meal_types = {"lunch", "dinner"}
user_avoid_ingredients = {"Beef", "Beef Jus", "Beef Sausage", "Beef Stock", "Bresaola Beef", "Cod (white fish)", "Cod Fish", "Cuttlefish", "Fish Sauce", "Gochujang Paste", "Gochujang Sauce", "Lamb", "Local Wild Fish", "Nile Perch", "Salmon", "Sea Bass", "Shrimps", "Squid", "Tuna", "White Fish", "Worcestershire Sauce"}
user_dislikes = {}
query = "Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Herb Mashed Potato, Baked Protein & Mashed Potato, Musakhan & Cauli Rice, Thai Mango Salad, Balkan Mushroom Rice, Leek & Potato Mash. Highest Rated Dishes: Balkan Mushroom Rice, Leek & Potato Mash, Chicken a la King & Roasted Sweet Potato, Burrito Bowl"
meal_plan, final_docs = generate_meal_plan(
    vectorstore,
    user_avoid_ingredients,
    user_dislikes,
    query,
    user_likes,
    user_pref,
    size,
    protein_option,
    protein_category,
    meal_types
)

# Print the meal plan
print(f"\nMeal Plan for User ID: 64a3930ab0d7b763fc053648:\n{meal_plan}\n")
time.sleep(4)  # Maintains RPM limit

final [{'allergens': ['Gluten'], 'carb': 53.0, 'cuisine': 'Arabic', 'description': 'Middle Eastern layered rice with vegetables. Contains Gluten.', 'dish_name': 'Maqluba', 'dish_type': ['Rice'], 'fat': 22.0, 'ingredients': ['Cumin Powder', 'Vegetable Stock', 'Eggplant', 'Sugar', 'Tomato', 'Olive Oil', 'Basmati Rice', 'Garlic', 'Cherry Tomato', 'Seven Spice Powder', 'Dill', 'Onion', 'Coriander Powder', 'Cauliflower', 'Chicken', 'Potatoes', 'Harissa Chilli Paste', 'Water', 'Salt', 'Black Pepper'], 'kcal': 698.0, 'meal_category': 'meal', 'protein': 70.0, 'protein_category': 'low', 'protein_option': ['Chicken'], 'recipe_id': '668e256a3d6d34934a26f047', 'size': 'extra_large', 'spice_level': 'Low'}, {'allergens': ['Dairy', 'Mustard'], 'carb': 37.0, 'cuisine': 'Indian', 'description': 'Indian inspired tikka with zucchini. Contains Dairy.', 'dish_name': 'Tikka Protein with Saffron Rice', 'dish_type': ['Rice'], 'fat': 24.0, 'ingredients': ['Chat Masala', 'Zucchini', 'Mint Leaves', 'Cumin Powder

### **user_2**

In [17]:
def fetch_json_data(file_path, output_file_path="cust_data.json"):
    """
    Fetches recipe data from a local JSON file, processes it, and returns only the relevant fields.
    """
    try:
        with open(file_path, 'r') as file:
            data = json.load(file)

        all_recipes = []

        # Iterate over each entry in the JSON file
        for entry in data:
            # Skip if entry is not a dictionary
            if not isinstance(entry, dict):
                print(f"Skipping non-dictionary entry: {entry}")
                continue

            recipe_info_list = entry.get("recipe_info", [])  # Ensure it's a list

            # Ensure recipe_info_list is actually a list before proceeding
            if not isinstance(recipe_info_list, list):
                print(f"Skipping entry due to unexpected format: {entry}")
                continue

            # Extract delivery date - now much simpler
            delivery_date = None
            entry_id = entry.get("_id", {})
            if isinstance(entry_id, dict):
                delivery_date = entry_id.get("delivery_date")
                # You might want to validate the date format here if needed

            # Iterate through each recipe_info dictionary in the list
            for recipe_info in recipe_info_list:
                # Skip if recipe_info is not a dictionary
                if not isinstance(recipe_info, dict):
                    print(f"Skipping non-dictionary recipe_info: {recipe_info}")
                    continue

                # Extract ingredients and merge with variant ingredients
                ingredients = recipe_info.get("ingredients", []) or []  # Ensure it's always a list
                variant_ingredients = []

                # Process the variants if available
                variants = recipe_info.get("variants", {})
                if isinstance(variants, dict):  # Ensure "variants" is a dictionary
                    variant_ingredients = variants.get("variant_ingredients", []) or []  # Ensure it's always a list

                # Merge ingredients from both the recipe_info and variants
                all_ingredients = list(set(ingredients + variant_ingredients))

                # Prepare the recipe data with only the necessary fields
                recipe = {
                    "dish_name": recipe_info.get("dish_name"),
                    "meal_category": recipe_info.get("meal_category"),
                    "description": recipe_info.get("description"),
                    "cuisine": recipe_info.get("cuisine"),
                    "ingredients": all_ingredients,  # Merged ingredients
                    "allergens_contain": recipe_info.get("allergens_contain", []),
                    "meal_type": entry.get("meal_type"),
                    "spice_level": recipe_info.get("spice_level", ""),
                    "is_auto_select": recipe_info.get("is_auto_select"),
                    "rating": recipe_info.get("rating") if "rating" in recipe_info else None,
                    "delivery_date": delivery_date  # Use the date string directly
                }

                # Add the processed recipe data to the list
                all_recipes.append(recipe)

        # If an output file path is provided, save the processed data to that file
        if output_file_path and all_recipes:
            with open(output_file_path, 'w') as output_file:
                json.dump(all_recipes, output_file, indent=4)
            print(f"Processed data saved to {output_file_path}")

        return all_recipes

    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return []
    except json.JSONDecodeError:
        print(f"Error: Failed to decode JSON data in file '{file_path}'. Please check the JSON syntax.")
        return []
    except Exception as e:
        print(f"Error processing file {file_path}: {str(e)}")
        return []

In [18]:
import json
input_file_path = "../user_data/prev_data/64a7adf4144bc0a5eb914149.json"
output_file_path = "processed_64a7adf4144bc0a5eb914149.json"

processed_data = fetch_json_data(input_file_path, output_file_path)


Processed data saved to processed_64a7adf4144bc0a5eb914149.json


In [19]:
df, query, user_dislikes = process_and_analyze_json("processed_64a7adf4144bc0a5eb914149.json")  # Call function for each file


In [22]:
query


'Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Mushroom Soup , Sticky Mango & Coconut Rice Pudding , Chocolate Muffin , Tomato Basil Soup, Oatmeal Apple Pancake, Coconut Curry and Quinoa '

In [26]:
user_pref = "Mediterranean"
user_likes = "Mushroom Soup , Sticky Mango & Coconut Rice Pudding , Chocolate Muffin , Tomato Basil Soup, Oatmeal Apple Pancake, Coconut Curry and Quinoa"
size = "medium"
protein_category = "low"
protein_option = ""
meal_types = {"lunch"}
user_avoid_ingredients = {"Lamb", "Beef Sausage", "Beef Stock", "Beef", "Beef Jus", "Bresaola Beef"}

meal_plan, final_docs = generate_meal_plan(
    vectorstore,
    user_avoid_ingredients,
    user_dislikes,
    query,
    user_likes,
    user_pref,
    size,
    protein_option,
    protein_category,
    meal_types
)

# Print the meal plan
print(f"\nMeal Plan for User ID: 64a7adf4144bc0a5eb914149:\n{meal_plan}\n")
time.sleep(4)  # Maintains RPM limit

Fetched recipes per meal type:
Meal: 10

Meal Plan for User ID: 64a7adf4144bc0a5eb914149:
```json
{
  "Monday": {
    "lunch": "Coconut Curry and Quinoa - Shrimps - Mediterranean - Quinoa"
  },
  "Tuesday": {
    "lunch": "Asian Meatballs with Fried Rice - Chicken - Asian - Potato"
  },
  "Wednesday": {
    "lunch": "Souvlaki & Quinoa Pilaf - Chicken - Mediterranean - Quinoa"
  },
  "Thursday": {
    "lunch": "Gyro Bowl - Chicken - Fusion - Chef's Choice"
  },
  "Friday": {
    "lunch": "Creamy Quinoa Bowl - Shrimps - European - Quinoa"
  },
  "Saturday": {
    "lunch": "Greek Chicken with Herb Rice - Chicken - European - Rice"
  },
  "Sunday": {
    "lunch": "Shawarma Bowl - Chicken - Arabic - Potato"
  }
}
```




#### **user_57**

In [27]:
import json
input_file_path = "6768aaa95ea91f54c85e8571.json"
output_file_path = "processed_6768aaa95ea91f54c85e8571.json"

processed_data = fetch_json_data(input_file_path, output_file_path)

Processed data saved to processed_6768aaa95ea91f54c85e8571.json


In [28]:
df, query, user_dislikes = process_and_analyze_json("processed_6768aaa95ea91f54c85e8571.json")  # Call function for each file


In [29]:
query


'Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Double Fried Eggs with Sausage Hash , Berry Smoothie , Grapes Pot, Acai Smoothie, Satay & Cauli Rice, Gruyere Omelette'

In [32]:
user_pref = "Mediterranean"
user_likes = "Double Fried Eggs with Sausage Hash , Berry Smoothie , Grapes Pot, Acai Smoothie, Satay & Cauli Rice, Gruyere Omelette"
size = "extra_large"
protein_category = "low"
protein_option = ""
meal_types = {"dinner", "breakfast"}
user_avoid_ingredients = {}

meal_plan, final_docs = generate_meal_plan(
    vectorstore,
    user_avoid_ingredients,
    user_dislikes,
    query,
    user_likes,
    user_pref,
    size,
    protein_option,
    protein_category,
    meal_types
)

# Print the meal plan
print(f"\nMeal Plan for User ID: 64a7adf4144bc0a5eb914149:\n{meal_plan}\n")
time.sleep(4)  # Maintains RPM limit

Fetched recipes per meal type:
Breakfast: 5
Meal: 10

Meal Plan for User ID: 64a7adf4144bc0a5eb914149:
```json
{
  "Monday": {
    "breakfast": "Double Fried Eggs with Sausage Hash - Chicken - Mediterranean - Comfort Food",
    "dinner": "Asian Meatballs with Fried Rice - Chicken - Asian - Potato"
  },
  "Tuesday": {
    "breakfast": "Mfarakeh - Chicken - Mediterranean - Potato",
    "dinner": "Greek Chicken with Herb Rice - Chicken - European - Rice"
  },
  "Wednesday": {
    "breakfast": "Breakfast Power Bowl - Chicken - Mediterranean - Chef's Choice",
    "dinner": "Souvlaki & Quinoa Pilaf - Beef - Mediterranean - Quinoa"
  },
  "Thursday": {
    "breakfast": "Stuffed Omelette & Roasted Peppers - Standard - Mediterranean - Comfort Food",
    "dinner": "Gyro Bowl - Beef - Fusion - Chef's Choice"
  },
  "Friday": {
    "breakfast": "Feta-Eggplant & Tomato Bake - Standard - Comfort Food - Comfort Food",
    "dinner": "Shawarma Bowl - Chicken - Arabic - Potato"
  },
  "Saturday": {
    

In [22]:
import os
from dotenv import load_dotenv
from langchain.vectorstores import Pinecone as LangChainPinecone
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai
import json
# from recipe_filter import filter_allergens_in_variants, filter_and_sort_recipes
import ast

# Load environment variables
load_dotenv(override=True)

gemini_api_key = os.getenv('GOOGLE_API_KEY')

# Ensure your Google API key is set
genai.configure(api_key=gemini_api_key)

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name = "recipes-fin"

# Load the embedding model (same as used for storing data)
embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Connect Pinecone to LangChain
vectorstore = LangChainPinecone(pc.Index(index_name), embed_model, text_key="text")

# Initialize ChatGoogleGenerativeAI for gemini-1.5-flash
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# Function to generate responses
def generate_response(prompt):
    model = genai.GenerativeModel("gemini-1.5-flash")
    # Generate content based on the prompt
    response = model.generate_content(prompt)
    return response.text

# Function to filter and sort recipes
def filter_recipes(vectorstore, user_avoid_ingredients, user_dislikes, query, meal_category, size, protein_option, protein_category, top_k):
    pinecone_filter = {
        "meal_category": {"$eq": meal_category},  # Filter for specific meal type
        "size": {"$eq": size},  # Filter for specific size
        # "protein_option": {"$eq": protein_option},  # Filter for specific protein option
        "protein_category": {"$eq": protein_category},  # Filter for specific protein category
        "allergens": {"$nin": list(user_avoid_ingredients)},  # Exclude recipes containing allergens
        "ingredients": {"$nin": list(user_avoid_ingredients)}  # Exclude recipes containing allergens
    }
    # # Add `protein_category` filter only if it's not empty
    # if protein_category:
    #     pinecone_filter["protein_category"] = {"$eq": protein_category}


    # Fetch documents using similarity_search
    docs = vectorstore.similarity_search(
        query=query,
        k=top_k * 3 ,  # Get extra to account for duplicates
        filter=pinecone_filter
    )

    # Merge recipes by recipe_id and combine protein_options
    merged_recipes = {}

    
    for doc in docs:
        metadata = doc.metadata
        recipe_id = metadata.get("recipe_id")
        
        # Skip if no recipe_id exists
        if not recipe_id:
            continue
        
        # Initialize new entry if recipe_id not seen
        if recipe_id not in merged_recipes:
            merged_recipes[recipe_id] = {
                **metadata,  # Copy all metadata
                "protein_option": {metadata.get("protein_option")}  # Start as set
            }
        else:
            # Merge protein_options
            existing = merged_recipes[recipe_id]
            new_protein = metadata.get("protein_option")
            if new_protein:
                existing["protein_option"].add(new_protein)
    
    # Prepare final output (convert sets to lists)
    final_recipes = [
        {
            **data,
            "protein_option": list(data["protein_option"]) if data["protein_option"] else []
        }
        for data in merged_recipes.values()
    ]
    print("final",final_recipes[:top_k])
    return final_recipes[:top_k]

# Function to format the filtered recipes into a structured meal plan prompt
def format_meal_plan_prompt(merged_recipes, query, user_avoid_ingredients, user_likes, user_dislikes, user_pref, meal_types, num_days):

        # Normalize meal types by stripping whitespace and converting to lowercase
    normalized_meal_types = {meal.strip().lower() for meal in meal_types}
    
    # Define the standard meal type order (customize as needed)
    STANDARD_ORDER = ['morning_snack', 'breakfast', 'lunch', 'dinner', 'evening_snack']
    
    # Filter and order the meal types based on standard order
    ordered_meal_types = [meal for meal in STANDARD_ORDER 
                         if meal in normalized_meal_types]
    
    prompt = f"""Generate a {num_days} day meal plan in JSON format using ONLY the provided recipes. Follow these rules exactly:

1. Recipe Usage:
- Use recipes exactly as provided - do not modify or create new ones
- Format each meal as: "<Dish Name> - <Selected Protein> - <Cuisine> - <Dish Type>"
- Use ONLY the dish_type field for the last component (never meal_category)
- For recipes with multiple protein options:
    * Ensure protein variety across the week (don't serve chicken 3 days in a row)

3. Meal Diversity:
- Alternate between:
  * Light vs heavy meals (e.g. salad → hearty stew)
  * Different cuisines (don't repeat back-to-back)
  * Cooking methods (grilled, baked, fried, etc.)
- Ensure no two consecutive meals have:
  * The same primary ingredient
  * Similar textures/flavor profiles

2. Meal Assignment:
- Never repeat recipes before all are used once
- Fill all selected meal slots - no empty values
- **Strictly follow meal categories:**
    * Breakfast: only 'breakfast' recipes having meal_category as breakfast
    * Lunch/Dinner: only 'meal' recipes having meal_category as meal
    * evening_snavk/morning_snack: only 'snack' recipes i.e. recipes having meal_category as snack
- Include only these meal types: {meal_types}

3. Daily Structure:
- You MUST include these meal types in EXACTLY this order: {ordered_meal_types}
- Never skip or rearrange these meal types
- Never include meal types not in this list
- Maintain consistent meal types across all days

Output Format: Present the meal plan as a JSON object where each day contains meal types as keys and the formatted meal string as values, like this example for Monday: {{\"Monday\": {{\"breakfast\": \"Dish Name - Protein - Cuisine - Dish Type\", \"lunch\": \"...\"}}}}User Preferences:
- Allergens: {user_avoid_ingredients}
- Likes: {user_likes}
- Dislikes: {user_dislikes}
- Preferred Dishes: {user_pref}
- Selected Meal Types: {meal_types}

Available Recipes:"""
    

    for i, recipe in enumerate(merged_recipes, 1):
        prompt += f"Meal {i}:\n"
        prompt += f" Dish Name: {recipe.get('dish_name', 'Unknown')}\n"
        prompt += f" Description: {recipe.get('description', 'No description')}\n"
        prompt += f" Protein Options: {', '.join(recipe.get('protein_option', []))}\n"  # Changed to handle list
        prompt += f" Dish Type: {', '.join(recipe.get('dish_type', []))}\n"  # Changed to handle list
        prompt += f" Ingredients: {', '.join(recipe.get('ingredients', []))}\n"
        prompt += f" Spice Level: {recipe.get('spice_level', 'Not specified')}\n"
        prompt += f" Cuisine: {recipe.get('cuisine', 'Unknown')}\n"
        prompt += f" Meal Category: {recipe.get('meal_category', 'Unknown')}\n"
    return prompt

# Main function to generate the meal plan
def generate_meal_plan(vectorstore, user_avoid_ingredients, user_dislikes, query, user_likes, user_pref, size, protein_option, protein_category, meal_types, num_days):
    # Define mapping of meal types to their respective counts
    meal_counts = {
        "breakfast": num_days,
        "snack": num_days,  # Each snack type (morning/evening) adds 8
        "meal": 0    # Lunch and Dinner are combined into "meal"
    }

    # Initialize counts
    total_snack_count = 0
    total_meal_count = 0
    fetched_recipes = {}

    # Calculate needed recipes
    if "morning_snack" in meal_types or "evening_snack" in meal_types:
        total_snack_count = meal_counts["snack"] * sum(1 for meal in meal_types if "snack" in meal)

    if "lunch" in meal_types:
        total_meal_count += num_days
    if "dinner" in meal_types:
        total_meal_count += num_days

    # Initial fetch (with allergens)
    for meal_type in meal_types:
        count = meal_counts.get(meal_type, 0)
        if count > 0:
            meal_size = "standard" if meal_type in ["breakfast", "snack"] else size
            fetched_recipes[meal_type] = filter_recipes(
                vectorstore, user_avoid_ingredients, user_dislikes, query, 
                meal_type, meal_size, "", protein_category, (count+3)
            )

    if total_meal_count > 0:
        fetched_recipes["meal"] = filter_recipes(
            vectorstore, user_avoid_ingredients, user_dislikes, query,
            "meal", size, protein_option, protein_category, (total_meal_count+3)
        )

    if total_snack_count > 0:
        fetched_recipes["snack"] = filter_recipes(
            vectorstore, user_avoid_ingredients, user_dislikes, query,
            "snack", "standard", "", protein_category, (total_snack_count+3)
        )

    # # Calculate expected minimums
    # expected_breakfast = max(0, meal_counts.get("breakfast", 0) - 4) if "breakfast" in meal_types else 0
    # expected_snack = max(0, total_snack_count - (3 if total_snack_count == 8 else 6 if total_snack_count == 16 else 0)) if any("snack" in mt for mt in meal_types) else 0
    # expected_meal = max(0, total_meal_count - (2 if total_meal_count == 8 else 7 if total_meal_count == 20 else 0)) if ("lunch" in meal_types or "dinner" in meal_types) else 0
    # print("fetched",fetched_recipes)    
    
    expected_breakfast = max((min(num_days, 4) + min(1, num_days)) - (0 if num_days <= 2 else 1 if num_days <= 5 else min(2, (min(num_days, 4) + min(1, num_days)) // 3)), 1) if "breakfast" in meal_types else 0
    expected_snack = max(total_snack_count - (0 if num_days <= 3 else 1 if num_days <= 5 else min(2, total_snack_count // 3)), 1) if any("snack" in mt for mt in meal_types) else 0
    expected_meal = max(total_meal_count - (0 if num_days <= 3 else 1 if num_days <= 5 else min(2, total_meal_count // 3)), 1) if ("lunch" in meal_types or "dinner" in meal_types) else 0    
    
    print("fetched",fetched_recipes)


    # Selective retry (only replaces deficient categories)
    if "breakfast" in meal_types and len(fetched_recipes.get("breakfast", [])) < expected_breakfast:
        print(f"Breakfast shortage ({len(fetched_recipes.get('breakfast', []))}/{expected_breakfast}), retrying without filters...")
        fetched_recipes["breakfast"] = filter_recipes(
            vectorstore, set(), user_dislikes, query,
            "breakfast", "standard", "", protein_category, meal_counts["breakfast"]+3
        )

    if any("snack" in mt for mt in meal_types) and len(fetched_recipes.get("snack", [])) < expected_snack:
        print(f"Snack shortage ({len(fetched_recipes.get('snack', []))}/{expected_snack}), retrying without filters...")
        fetched_recipes["snack"] = filter_recipes(
            vectorstore, set(), user_dislikes, query,
            "snack", "standard", "", protein_category, total_snack_count+3
        )

    if ("lunch" in meal_types or "dinner" in meal_types) and len(fetched_recipes.get("meal", [])) < expected_meal:
        print(f"Meal shortage ({len(fetched_recipes.get('meal', []))}/{expected_meal}), retrying without filters...")
        fetched_recipes["meal"] = filter_recipes(
            vectorstore, set(), user_dislikes, query,
            "meal", size, protein_option, protein_category, total_meal_count+3
        )
    print("fetched",fetched_recipes)

    # Debug output
    print("\nFinal recipe counts:")
    for category in ["breakfast", "snack", "meal"]:
        if category in fetched_recipes:
            print(f"{category.capitalize()}: {len(fetched_recipes[category])}")

    # Generate meal plan
    final_docs = [recipe for recipes in fetched_recipes.values() for recipe in recipes]
    final_prompt = format_meal_plan_prompt(final_docs, query, user_avoid_ingredients, user_likes, user_dislikes, user_pref, meal_types, num_days)
    meal_plan = generate_response(final_prompt)

    return meal_plan, final_docs

# Example usage
if __name__ == "__main__":
    # User preferences (replace with dynamic input if needed)
    user_avoid_ingredients = {}  # Example allergens
    # user_dislikes = {
    #     "Cod (white fish)", "Cod Fish", "Cuttlefish", "Fish Sauce",
    #     "Gochujang Paste", "Gochujang Sauce", "Local Wild Fish",
    #     "Nile Perch", "Salmon", "Sea Bass", "Squid", "Tuna",
    #     "White Fish", "Worcestershire Sauce"
    # }  # Example disliked ingredients
    
    # query = "Spice Level: Medium, Cuisine: Mediterranean,European,Comfort Food. Popular Dishes: Classic Chicken Salad, Crudites & Sour Cream Dip, Mini Quiches, Omega Egg Protein Pot"

    # Generate the meal plan
    # generate_meal_plan(vectorstore, user_avoid_ingredients, user_dislikes, query, user_likes, user_pref)

In [27]:
import time

user_pref = "Mediterranean"
user_likes = "Balkan Mushroom Rice, Leek & Potato Fusilli Pasta, Butter Masala & Rice , Kale & Quinoa Salad"
size = "large"
protein_category = "balance"
protein_option = ""
meal_types = {"dinner", "lunch", "evening_snack"}
user_avoid_ingredients = {"Beef", "Beef Jus", "Beef Sausage", "Beef Stock", "Bresaola Beef", "Brioche Bread", "Chicken", "Chicken Jus", "Chicken Sausage", "Chicken Stock", "Eggs", "Lamb", "Lasagna", "Mayonnaise", "Panettone Bread", "Tagliatelle", "Turkey"}
# user_avoid_ingredients = {}
num_days = 6
user_dislikes = {}
query = "Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Balkan Mushroom Rice, Leek & Potato Fusilli Pasta, Butter Masala & Rice , Kale & Quinoa Salad"
meal_plan, final_docs = generate_meal_plan(
    vectorstore,
    user_avoid_ingredients,
    user_dislikes,
    query,
    user_likes,
    user_pref,
    size,
    protein_option,
    protein_category,
    meal_types,
    num_days
)

# Print the meal plan
print(f"\nMeal Plan for User ID: 64a7adf4144bc0a5eb914149:\n{meal_plan}\n")
time.sleep(4)  # Maintains RPM limit

final [{'allergens': ['Soy', 'Mustard', 'Dairy'], 'carb': 47.0, 'cuisine': 'European', 'description': 'Crunchy veggies & feta cheese with olive oil vinaigrette dressing. Contains Dairy.', 'dish_name': 'Greek Salad', 'dish_type': ['Salad'], 'fat': 28.0, 'ingredients': ['White Vinegar', 'Edamame', 'Oregano', 'Coriander Seeds', 'Black Olives', 'Sugar', 'Chickpeas', 'Olive Oil', 'Cherry Tomato', 'Garlic', 'Onion', 'Green Olives', 'Bay Leaf', 'Cucumber', 'Black Pepper', 'Star Anise', 'Romaine Lettuce', 'Yellow Mustard', 'Broccoli', 'Feta Cheese', 'Beetroot', 'Water', 'Salt', 'Honey', 'Lime'], 'kcal': 541.0, 'meal_category': 'meal', 'protein': 24.0, 'protein_category': 'balance', 'protein_option': ['Vegetarian'], 'recipe_id': '667e02ec013b5ad7ef2909d4', 'size': 'large', 'spice_level': 'Low'}, {'allergens': ['Dairy', 'Gluten'], 'carb': 70.0, 'cuisine': 'Mediterranean', 'description': 'In a creamy sauce with sautéed mushrooms. Contains Dairy & Gluten.', 'dish_name': 'Fusilli Alfredo', 'dish_ty